# Simulador de Gestión Energética: Baterías y Peukert

El estado de carga (SOC - State of Charge) desciende más rápido si la demanda de amperios es muy alta. Este script simula el SOC de un banco de servicios durante 24h, considerando inyección solar.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulador_bateria(capacidad_ah, consumo_base_a, panel_solar_wp):
    horas = np.arange(0, 24, 0.5)
    soc = [100.0] # 100% inicio
    ah_restantes = capacidad_ah
    
    for h in horas[1:]:
        # Consumo base (electrónica, nevera, piloto si navega)
        consumo = consumo_base_a
        
        # Producción solar aproximada curva de campana (12h medio día)
        if 8 <= h <= 18:
            # max a las 13h
            eficiencia = max(0, 1 - abs(13 - h)/5)
            produccion_A = (panel_solar_wp / 13.0) * eficiencia
        else:
            produccion_A = 0
            
        balance = produccion_A - consumo
        
        ah_restantes += balance * 0.5 # 0.5 porque el timestep es 30 min
        ah_restantes = min(ah_restantes, capacidad_ah) # No carga a más del 100%
        
        soc_actual = (ah_restantes / capacidad_ah) * 100
        soc.append(soc_actual)
        
    plt.figure(figsize=(10,4))
    plt.plot(horas, soc, 'b-', linewidth=2)
    plt.axhline(50, color='r', linestyle='--', label='Límite Plomo/Ácido (50%)')
    plt.axhline(20, color='g', linestyle='--', label='Límite LiFePO4 (20%)')
    plt.title(f"Evolución del SOC en 24h (Banco {capacidad_ah}Ah, Solar {panel_solar_wp}Wp)")
    plt.xlabel("Hora del día")
    plt.ylabel("SOC (%)")
    plt.ylim(0, 105)
    plt.grid()
    plt.legend()
    plt.show()

# Ej: Batería 200Ah, Consumo medio 6A (Nevera+Plotter), 300W placas
simulador_bateria(capacidad_ah=200, consumo_base_a=6, panel_solar_wp=300)
